# Review sentiment with multilingual DistilBERT (run on Google Colab, GPU runtime)

Fine-tunes `distilbert-base-multilingual-cased` to flag negative Olist reviews (score 1-2) from Portuguese text, and compares with the TF-IDF baseline in `ml/review_sentiment.py` (F1 0.826).

1. Runtime → Change runtime type → **T4 GPU**
2. Upload `olist_order_reviews_dataset.csv` (from Kaggle) when prompted
3. Run all cells; copy the final F1 into `ml/reports/review_sentiment_transformer.json`

In [ ]:
!pip -q install transformers datasets accelerate scikit-learn

In [ ]:
from google.colab import files
uploaded = files.upload()  # choose olist_order_reviews_dataset.csv

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
df = pd.read_csv('olist_order_reviews_dataset.csv')
df = df[df.review_comment_message.notna()].copy()
df['text'] = (df.review_comment_title.fillna('') + ' ' + df.review_comment_message).str.strip()
df = df[df.text.str.len() >= 3]
df['label'] = (df.review_score <= 2).astype(int)
train, test = train_test_split(df[['text', 'label']], test_size=0.2, stratify=df.label, random_state=0)
len(train), len(test), df.label.mean()

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
MODEL = 'distilbert-base-multilingual-cased'
tok = AutoTokenizer.from_pretrained(MODEL)
def encode(batch):
    return tok(batch['text'], truncation=True, max_length=128)
train_ds = Dataset.from_pandas(train, preserve_index=False).map(encode, batched=True)
test_ds = Dataset.from_pandas(test, preserve_index=False).map(encode, batched=True)

In [ ]:
import numpy as np
from sklearn.metrics import f1_score
model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=2)
def metrics(p):
    return {'f1_negative': f1_score(p.label_ids, np.argmax(p.predictions, axis=1))}
args = TrainingArguments('out', num_train_epochs=2, per_device_train_batch_size=32, per_device_eval_batch_size=64,
                         learning_rate=3e-5, eval_strategy='epoch', save_strategy='no', fp16=True, report_to=[])
trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=test_ds, tokenizer=tok, compute_metrics=metrics)
trainer.train()
result = trainer.evaluate()
result

In [ ]:
# Optional: publish to the Hugging Face Hub (needs your HF token)
# from huggingface_hub import notebook_login; notebook_login()
# trainer.push_to_hub('olist-review-negative-distilbert')